# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, which loads and processes datasets conformant with the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

Let's list all record sets and their fields (`@id`s).

In [ ]:
# List all record sets, fields, and columns by their @id
print("Available record sets (by @id):")
record_sets_info = []
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set.id} | name: {record_set.name}")
    fields_ids = []
    for field in record_set.fields:
        fields_ids.append(field.id)
        print(f"    - Field @id: {field.id} | name: {field.name}")
        # Show columns inside the field if present
        if hasattr(field, 'columns'):
            for column in field.columns:
                print(f"        - Column @id: {column.id} | name: {column.name}")
    record_sets_info.append({
        'id': record_set.id,
        'name': record_set.name,
        'fields': fields_ids
    })

# For convenience, collect all record_set @id for later extraction
available_recordset_ids = [rs['id'] for rs in record_sets_info]

## 3. Data Extraction

Load data from each record set into a `pd.DataFrame` for analysis. We reference record set and field `@id`s from the previous section. If there are multiple record sets, we load all; otherwise, just the main one.

In [ ]:
dataframes = {}

for recordset_id in available_recordset_ids:
    print(f'Loading records from RecordSet @id: {recordset_id}')
    records = list(dataset.records(record_set=recordset_id))
    df = pd.DataFrame(records)
    dataframes[recordset_id] = df
    print(f'  Loaded {len(df)} records, columns: {df.columns.tolist()}')
    print()

# Display columns for the first record set as an example
if available_recordset_ids:
    first_recordset_id = available_recordset_ids[0]
    print(f"Columns in the first record set (@id: {first_recordset_id}): \n{dataframes[first_recordset_id].columns.tolist()}")
    display(dataframes[first_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (by `@id`) and perform typical analysis: filtering, normalization, and grouping. Please update the selected `numeric_field_id` and `group_field_id` for your use-case, based on the earlier output.

For demonstration:
  - We'll pick a numeric field (e.g., age or interval in months, as available).
  - We'll use a group field such as a categorical clinical feature (e.g., sex, MSI status).

In [ ]:
# Please update these IDs as appropriate based on your data
numeric_field_id = None
group_field_id = None
# Automatically attempt to pick typical field ids
possible_numeric = ['cr:age', 'cr:IntervalBetweenCancers_Months', 'cr:interval_months', 'cr:interval', 'cr:years', 'cr:Age', 'cr:Interval']
possible_group = ['cr:sex', 'cr:Sex', 'cr:msi_status', 'cr:MSI_status', 'cr:MSI', 'cr:msi', 'cr:second_primary_site', 'cr:PrimaryCancerSite']

first_df = dataframes[first_recordset_id]

# Try to select a numeric field automatically
for candidate in possible_numeric:
    if candidate in first_df.columns:
        numeric_field_id = candidate
        break
if numeric_field_id is None:
    # Default to the first numeric column
    for col in first_df.columns:
        if pd.api.types.is_numeric_dtype(first_df[col]):
            numeric_field_id = col
            break

# Try to select a group field automatically
for candidate in possible_group:
    if candidate in first_df.columns:
        group_field_id = candidate
        break
if group_field_id is None:
    # Default to first object/categorical column
    for col in first_df.columns:
        if pd.api.types.is_object_dtype(first_df[col]):
            group_field_id = col
            break

print(f"EDA will use numeric field: {numeric_field_id}")
print(f"EDA will use group field: {group_field_id}")

if numeric_field_id in first_df.columns:
    # Remove outliers: filter by values above the mean
    threshold = first_df[numeric_field_id].mean()
    filtered_df = first_df[first_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group and aggregate
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field detected for EDA. Please check columns above.")

## 5. Visualization

Visualize field distributions and groupings in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
if numeric_field_id in first_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(first_df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field (if available)
    if group_field_id in first_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=first_df[group_field_id], y=first_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- We loaded and explored the FAIR² clinical dataset via its Croissant schema using the `mlcroissant` library, referencing all entities by their `@id`s.
- We reviewed available record sets, fields, and columns and extracted records for analysis in pandas.
- Exploratory analysis and basic visualizations show the data's structure, enabling clinical/biostatistical questions such as the relationship between interval durations or age and categorical features like MSI status.

**Next steps:**
- Apply hypothesis testing, advanced modeling, or more detailed clinical outcome analyses as required for your research.